# Mythos 6 — Kaggle training notebook

Thin wrapper around the `mythos6` repo's `scripts/`/`src/mythos` pipeline. See `ARCHITECTURE.md` and `MILESTONES.md` in the repo for the design rationale and current status — read those before changing anything here.

**Known limitation:** `train.py` is single-GPU only (no DDP yet). If your Kaggle notebook accelerator is set to **2x T4**, this will only use one of them while Kaggle still bills your weekly quota for both — set the accelerator to a single **GPU T4 x1** instead until multi-GPU support lands (see MILESTONES.md). Session cap is 12h and disconnects happen; the loop checkpoints every `--save-every` steps and `--resume` picks back up from the last checkpoint, so re-running this notebook after a disconnect is safe.

In [ ]:
import os

REPO_URL = "<FILL IN: your git remote for the mythos6 repo>"
WORK_DIR = "/kaggle/working/mythos6"

if not os.path.exists(WORK_DIR):
    !git clone $REPO_URL $WORK_DIR
%cd $WORK_DIR
!pip install -q -r requirements.txt

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected — check the notebook's accelerator setting."
print(torch.cuda.get_device_name(0), "| bf16 supported:", torch.cuda.is_bf16_supported())
print("device count:", torch.cuda.device_count(), "(train.py only uses device 0 — see the limitation note above)")

## 1. Tokenizer
Train once, reuse across runs. Point `--n-docs` at whatever the current MILESTONES.md full-scale target is (the repo's local validation runs used a few thousand docs only — do not reuse that number here).

In [ ]:
TOKENIZER_PATH = "/kaggle/working/artifacts/tokenizer.json"
if not os.path.exists(TOKENIZER_PATH):
    !python scripts/train_tokenizer.py --n-docs 200000 --out $TOKENIZER_PATH

## 2. Contamination + dedup gate
Per ARCHITECTURE.md sec 5.3 this is a gate, not a formality — don't skip it before a real run.

In [ ]:
!python scripts/contamination_scan.py --n-corpus-docs 50000
!python scripts/dedup_scan.py --n-docs 50000

## 3. Pre-tokenize into packed shards
Writes to Kaggle's working directory (ephemeral — re-run if the session resets and shards are gone; this step is CPU-only, no GPU quota consumed).

In [ ]:
SHARD_DIR = "/kaggle/working/data/packed"
PRESET = "mythos6-130m-dense"  # start here per MILESTONES.md M2, not the 320m config
SEQ_LEN = 4096
N_SEQUENCES = 400000  # ~1.6B tokens; adjust against the actual token budget in ARCHITECTURE.md sec 6.3

if not os.path.exists(f"{SHARD_DIR}/manifest.json"):
    !python scripts/pretokenize.py --tokenizer $TOKENIZER_PATH --out-dir $SHARD_DIR \
        --seq-len $SEQ_LEN --n-sequences $N_SEQUENCES

## 4. Train
Re-run this cell with `--resume` after any disconnect — it's a no-op if there's nothing to resume from yet.

In [ ]:
RUN_DIR = "/kaggle/working/runs/mythos6-130m-dense"

!python -m mythos.train \
    --preset $PRESET \
    --data-dir $SHARD_DIR \
    --out-dir $RUN_DIR \
    --micro-batch-size 8 \
    --grad-accum-steps 16 \
    --max-steps 50000 \
    --save-every 500 \
    --resume

## 5. Persist checkpoints
`/kaggle/working` survives within a session but Kaggle deletes it on some resets — copy checkpoints to a Kaggle Dataset (or Drive, if mounted) before the session ends.

In [ ]:
# e.g.: !cp -r $RUN_DIR /kaggle/working/persisted_checkpoints/
# then create/version a Kaggle Dataset from /kaggle/working/persisted_checkpoints via the Kaggle UI or API.